In [15]:
from pyrosetta.rosetta.protocols.simple_moves.sidechain_moves import SidechainMCMover
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import (RestrictToRepacking,
    OperateOnResidueSubset, PreventRepackingRLT)
from pyrosetta.rosetta.core.select.residue_selector import ResidueIndexSelector
from pyrosetta import init, pose_from_pdb, get_fa_scorefxn
from pyrosetta.rosetta.numeric.random import rg
import multiprocessing as mp

init()
pose = pose_from_pdb("4oy7_rosetta_in.pdb")
scorefxn = get_score_function()
nres = pose.total_residue()
for i in range(1, nres+1):
    if 'CU' in pose.residue(i).name().upper():
        print(f"Found Cu at residue {i}: {pose.residue(i).name()}")
        cu = i






┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

In [11]:
# freeze everything outside a 20 A radius of the Cu or 2.5 Å withib
frozen = []
for i in range(1, nres+1):
    if i != cu:
        for k in range(1, pose.residue(i).natoms()+1):
            if (pose.residue(i).xyz(k).distance(pose.residue(cu).xyz(1)) < 2.5):
                frozen.append(i)
                break
            elif (pose.residue(i).xyz(k).distance(pose.residue(cu).xyz(1)) > 20.0):
                frozen.append(i)
                break
print(frozen)
                

[1, 6, 7, 10, 13, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 44, 45, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 74, 78, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 110, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 146, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 189, 190, 191, 192, 193, 194, 195, 196]


In [12]:
# single operation

tf = TaskFactory()
tf.push_back(RestrictToRepacking())                       
tf.push_back(OperateOnResidueSubset(PreventRepackingRLT(),
             ResidueIndexSelector(','.join(map(str, frozen)))))

sc_mc = SidechainMCMover()
sc_mc.setup(scorefxn)         
sc_mc.set_ntrials(10000)
sc_mc.set_temperature(1.0)
sc_mc.set_task_factory(tf)
sc_mc.apply(pose)
pose.dump_pdb("scmc_out.pdb")

True

In [17]:
# embarrassingly parallelize the same operation with multiprocessing
n_cores = 4
n_replicas = 20

def worker(i):
    rg().set_seed("mt19937", 1000 + i)      
    tf = TaskFactory()
    tf.push_back(RestrictToRepacking())
    tf.push_back(OperateOnResidueSubset(PreventRepackingRLT(),
                 ResidueIndexSelector(','.join(map(str, frozen)))))
    sc = SidechainMCMover()
    sc.setup(scorefxn)                       
    sc.set_ntrials(10000)
    sc.set_temperature(1.0)
    sc.set_task_factory(tf)
    tmp_pose = pose.clone()
    sc.apply(tmp_pose)
    tmp_pose.dump_pdb(f"rep_{i}.pdb")               # results live on disk, not in the return value

with Pool(n_cores) as pool:
    pool.map(worker, range(n_replicas))
